# Exploración visual de `smoke_local_scraper.db`

Notebook aparte para inspeccionar la base de datos local con foco en análisis práctico:
- últimas entradas
- anuncios más antiguos y aún activos
- filtros por precio
- evolución de `price_first_seen` vs `price`
- vista rápida de `first_seen` y `last_seen`

## 1. Conexión y configuración

Abrimos la base SQLite local y dejamos listas algunas utilidades para reutilizar consultas en las siguientes celdas.

In [11]:
from pathlib import Path
import sqlite3
import pandas as pd
from IPython.display import display, Markdown

DB_PATH = Path("smoke_local_scraper.db")
if not DB_PATH.exists():
    raise FileNotFoundError(f"No se encuentra la base de datos: {DB_PATH.resolve()}")

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
properties_df = pd.read_sql_query("SELECT * FROM properties", conn)
print(f"Conectado a: {DB_PATH.resolve()}")
print(f"Filas en properties: {len(properties_df)}")

Conectado a: /Users/Adri/Desktop/Github_repos/Immo_scraper/smoke_local_scraper.db
Filas en properties: 173


## 2. Resumen rápido

Una panorámica mínima para saber qué datos tenemos antes de filtrar.

In [12]:
summary = pd.DataFrame({
    'metric': [
        'rows',
        'unique_property_id',
        'active_rows',
        'inactive_rows',
        'price_min',
        'price_median',
        'price_max',
        'price_first_seen_min',
        'price_first_seen_median',
        'price_first_seen_max',
    ],
    'value': [
        len(properties_df),
        properties_df['property_id'].nunique(dropna=True),
        (properties_df['status'] == 'active').sum() if 'status' in properties_df else None,
        (properties_df['status'] == 'inactive').sum() if 'status' in properties_df else None,
        properties_df['price'].min(),
        properties_df['price'].median(),
        properties_df['price'].max(),
        properties_df['price_first_seen'].min() if 'price_first_seen' in properties_df else None,
        properties_df['price_first_seen'].median() if 'price_first_seen' in properties_df else None,
        properties_df['price_first_seen'].max() if 'price_first_seen' in properties_df else None,
    ]
})
display(summary)
display(properties_df[['property_id', 'source', 'title', 'price', 'price_first_seen', 'first_seen', 'last_seen', 'status']].head(10))

,metric,value
0,rows,173.0
1,unique_property_id,173.0
2,active_rows,173.0
3,inactive_rows,0.0
4,price_min,45.0
5,price_median,800000.0
6,price_max,5300000.0
7,price_first_seen_min,45.0
8,price_first_seen_median,930000.0
9,price_first_seen_max,inf


,property_id,source,title,price,price_first_seen,first_seen,last_seen,status
0,amat_practica-i-accesible-planta-baixa-a-tocar...,amat,"Pràctica i accesible planta baixa, a tocar del...",310000.0,310000.0,2026-04-29T19:06:06+00:00,2026-05-09T14:21:20+00:00,active
1,amat_fantastica-casa-de-disseny-a-mirasol,amat,Fantàstica casa de disseny a Mirasol,1225000.0,1225000.0,2026-04-29T19:06:06+00:00,2026-05-09T14:21:20+00:00,active
2,amat_casa-a-sant-cugat-dos-habitatges-en-un,amat,Casa a Sant Cugat: Dos habitatges en un,845000.0,845000.0,2026-04-29T19:06:06+00:00,2026-04-29T19:08:50+00:00,active
3,amat_placa-daparcament-en-venda-al-centre,amat,Plaça d'aparcament en venda al centre,20000.0,20000.0,2026-04-29T19:06:06+00:00,2026-05-09T14:21:20+00:00,active
4,amat_magnifica-casa-al-golf-amb-grans-espais,amat,"Magnífica casa al Golf, amb grans espais",1575000.0,1575000.0,2026-04-29T19:06:06+00:00,2026-05-09T14:21:20+00:00,active
5,qgat_homes_ref-1717,qgat_homes,Casa amb vistes a Mirasol,1150000.0,1150000.0,2026-05-09T11:22:01+00:00,2026-05-09T11:53:37+00:00,active
6,qgat_homes_ref-1682,qgat_homes,Casa amb vistes a Mirasol,1150000.0,1150000.0,2026-05-09T11:22:01+00:00,2026-05-09T11:53:37+00:00,active
7,qgat_homes_ref-1716,qgat_homes,Àtic en venda a Sant Cugat del Vallès - Can Mates,615000.0,615000.0,2026-05-09T11:22:01+00:00,2026-05-09T11:53:37+00:00,active
8,qgat_homes_ref-1699,qgat_homes,Cases en venda a Sant Cugat del Vallès - Valld...,589000.0,589000.0,2026-05-09T11:22:01+00:00,2026-05-09T11:53:37+00:00,active
9,qgat_homes_ref-1685,qgat_homes,Casa adossada a Sant Domenech,775000.0,775000.0,2026-05-09T11:22:01+00:00,2026-05-09T11:53:37+00:00,active


## 3. Helpers de inspección

Detectamos automáticamente las columnas temporales y de precio inicial para que la notebook siga funcionando si el esquema cambia un poco.

In [13]:
properties_columns = pd.read_sql_query("PRAGMA table_info(properties)", conn)['name'].tolist()
first_seen_candidates = ['first_seen', 'first_seen_at', 'created_at', 'published_at', 'date']
last_seen_candidates = ['last_seen', 'last_seen_at', 'updated_at', 'scraped_at', 'date']
price_first_seen_col = 'price_first_seen' if 'price_first_seen' in properties_columns else None
first_seen_col = next((col for col in first_seen_candidates if col in properties_columns), None)
last_seen_col = next((col for col in last_seen_candidates if col in properties_columns), None)

def load_active_properties():
    if 'status' in properties_columns:
        return pd.read_sql_query("SELECT * FROM properties WHERE status = 'active'", conn)
    return properties_df.copy()

def _days_since(values: pd.Series) -> pd.Series:
    now_utc = pd.Timestamp.now(tz='UTC')
    parsed = pd.to_datetime(values, errors='coerce', utc=True)
    return ((now_utc - parsed).dt.total_seconds() // 86400).astype('Int64')

def add_age_columns(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()
    if first_seen_col and first_seen_col in result.columns:
        result['days_online'] = _days_since(result[first_seen_col])
    else:
        result['days_online'] = pd.Series([pd.NA] * len(result), dtype='Int64')
    if last_seen_col and last_seen_col in result.columns:
        result['days_since_last_seen'] = _days_since(result[last_seen_col])
    else:
        result['days_since_last_seen'] = pd.Series([pd.NA] * len(result), dtype='Int64')
    return result

def compact_cols(df: pd.DataFrame, columns: list) -> pd.DataFrame:
    existing = [col for col in columns if col in df.columns]
    return df[existing] if existing else df

## 4. Últimas entradas

Vista de los anuncios más recientes según `last_seen` o `first_seen`, según lo que exista en la base.

In [14]:
active_df = add_age_columns(load_active_properties())
sort_col = last_seen_col if last_seen_col in active_df.columns else first_seen_col
sort_col = sort_col if sort_col in active_df.columns else None

recent_columns = [
    'property_id', 'title', 'source', 'city', 'price', 'price_first_seen',
    'first_seen', 'last_seen', 'days_online', 'days_since_last_seen', 'status'
]

if sort_col:
    recent_df = active_df.sort_values(by=sort_col, ascending=False).head(20)
else:
    recent_df = active_df.head(20)

display(compact_cols(recent_df, recent_columns))

,property_id,title,source,city,price,price_first_seen,first_seen,last_seen,days_online,days_since_last_seen,status
172,aproperties_casa-de-lujo-de-573-m2-con-terraza...,Casa vanguardista en venta en El Golf-Can Trabal,aproperties,Sant Cugat del Vallès,2900000.0,2900000.0,2026-05-09T15:18:18+00:00,2026-05-09T15:18:18+00:00,0,0,active
150,aproperties_casa-de-452-m2-con-terraza-aparcam...,Casa en venta en Sant Cugat,aproperties,Sant Cugat del Vallès,2450000.0,2450000.0,2026-05-09T15:18:18+00:00,2026-05-09T15:18:18+00:00,0,0,active
148,aproperties_piso-de-76-m2-con-terraza-en-venta...,"Piso en venta en Centre-Estació, Sant Cugat de...",aproperties,Sant Cugat del Vallès,435000.0,435000.0,2026-05-09T15:18:18+00:00,2026-05-09T15:18:18+00:00,0,0,active
147,aproperties_casa-de-290-m2-con-terraza-aparcam...,Casa en venta en Arxius,aproperties,Sant Cugat del Vallès,1650000.0,1650000.0,2026-05-09T15:18:18+00:00,2026-05-09T15:18:18+00:00,0,0,active
146,aproperties_casa-de-374-m2-con-terraza-y-aparc...,Exclusiva casa en venta frente del Golf de San...,aproperties,Sant Cugat del Vallès,2950000.0,2950000.0,2026-05-09T15:18:18+00:00,2026-05-09T15:18:18+00:00,0,0,active
145,aproperties_obra-nueva-de-191-m2-con-terraza-y...,Vivir el lujo en una finca histórica en el cen...,aproperties,Sant Cugat del Vallès,1700000.0,1700000.0,2026-05-09T15:18:18+00:00,2026-05-09T15:18:18+00:00,0,0,active
144,aproperties_casa-de-593-m2-con-terraza-aparcam...,Casa en venta en Valldoreix,aproperties,Sant Cugat del Vallès,1890000.0,1890000.0,2026-05-09T15:18:18+00:00,2026-05-09T15:18:18+00:00,0,0,active
143,aproperties_parcela-de-1677-m2-en-venta-en-val...,Dos terrenos en venta en Valldoreix,aproperties,Sant Cugat del Vallès,1375000.0,1375000.0,2026-05-09T15:18:18+00:00,2026-05-09T15:18:18+00:00,0,0,active
142,aproperties_casa-de-lujo-de-347-m2-con-terraza...,Casa en venta en Valldoreix,aproperties,Sant Cugat del Vallès,1475000.0,1475000.0,2026-05-09T15:18:18+00:00,2026-05-09T15:18:18+00:00,0,0,active
141,aproperties_parcela-de-1462-m2-en-venta-en-val...,Terreno en venta en Valldoreix,aproperties,Sant Cugat del Vallès,520000.0,520000.0,2026-05-09T15:18:18+00:00,2026-05-09T15:18:18+00:00,0,0,active


## 5. Anuncios más antiguos y aún activos

Ordenados por antigüedad desde `first_seen`, útil para ver qué sigue vivo en la web desde hace más tiempo.

In [15]:
if first_seen_col and first_seen_col in active_df.columns:
    oldest_df = active_df.sort_values(by=['days_online', 'first_seen'], ascending=[False, True]).head(25)
else:
    oldest_df = active_df.head(25)

display(compact_cols(oldest_df, [
    'property_id', 'title', 'source', 'city', 'price', 'price_first_seen',
    'first_seen', 'last_seen', 'days_online', 'days_since_last_seen', 'status'
]))

,property_id,title,source,city,price,price_first_seen,first_seen,last_seen,days_online,days_since_last_seen,status
0,amat_practica-i-accesible-planta-baixa-a-tocar...,"Pràctica i accesible planta baixa, a tocar del...",amat,Sant Cugat del Vallès,310000.0,310000.0,2026-04-29T19:06:06+00:00,2026-05-09T14:21:20+00:00,9,0,active
1,amat_fantastica-casa-de-disseny-a-mirasol,Fantàstica casa de disseny a Mirasol,amat,Sant Cugat del Vallès,1225000.0,1225000.0,2026-04-29T19:06:06+00:00,2026-05-09T14:21:20+00:00,9,0,active
2,amat_casa-a-sant-cugat-dos-habitatges-en-un,Casa a Sant Cugat: Dos habitatges en un,amat,Sant Cugat del Vallès,845000.0,845000.0,2026-04-29T19:06:06+00:00,2026-04-29T19:08:50+00:00,9,9,active
3,amat_placa-daparcament-en-venda-al-centre,Plaça d'aparcament en venda al centre,amat,Sant Cugat del Vallès,20000.0,20000.0,2026-04-29T19:06:06+00:00,2026-05-09T14:21:20+00:00,9,0,active
4,amat_magnifica-casa-al-golf-amb-grans-espais,"Magnífica casa al Golf, amb grans espais",amat,Sant Cugat del Vallès,1575000.0,1575000.0,2026-04-29T19:06:06+00:00,2026-05-09T14:21:20+00:00,9,0,active
5,qgat_homes_ref-1717,Casa amb vistes a Mirasol,qgat_homes,Sant Cugat del Vallès,1150000.0,1150000.0,2026-05-09T11:22:01+00:00,2026-05-09T11:53:37+00:00,0,0,active
6,qgat_homes_ref-1682,Casa amb vistes a Mirasol,qgat_homes,Sant Cugat del Vallès,1150000.0,1150000.0,2026-05-09T11:22:01+00:00,2026-05-09T11:53:37+00:00,0,0,active
7,qgat_homes_ref-1716,Àtic en venda a Sant Cugat del Vallès - Can Mates,qgat_homes,Sant Cugat del Vallès,615000.0,615000.0,2026-05-09T11:22:01+00:00,2026-05-09T11:53:37+00:00,0,0,active
8,qgat_homes_ref-1699,Cases en venda a Sant Cugat del Vallès - Valld...,qgat_homes,Sant Cugat del Vallès,589000.0,589000.0,2026-05-09T11:22:01+00:00,2026-05-09T11:53:37+00:00,0,0,active
9,qgat_homes_ref-1685,Casa adossada a Sant Domenech,qgat_homes,Sant Cugat del Vallès,775000.0,775000.0,2026-05-09T11:22:01+00:00,2026-05-09T11:53:37+00:00,0,0,active


## 6. Filtros por precio

Cambia los valores `MIN_PRICE` y `MAX_PRICE` para acotar el rango que quieres inspeccionar.

In [16]:
MIN_PRICE = 0
MAX_PRICE = 700000
MIN_DAYS_ONLINE = 0

filtered_df = active_df.copy()
if 'price' in filtered_df.columns:
    filtered_df = filtered_df[filtered_df['price'].notna()]
    filtered_df = filtered_df[(filtered_df['price'] >= MIN_PRICE) & (filtered_df['price'] <= MAX_PRICE)]
if 'days_online' in filtered_df.columns:
    filtered_df = filtered_df[filtered_df['days_online'].fillna(-1) >= MIN_DAYS_ONLINE]

display(compact_cols(filtered_df.sort_values(by=['price', 'days_online'], ascending=[True, False]).head(50), [
    'property_id', 'title', 'source', 'city', 'price', 'price_first_seen',
    'first_seen', 'last_seen', 'days_online', 'days_since_last_seen', 'status'
]))

,property_id,title,source,city,price,price_first_seen,first_seen,last_seen,days_online,days_since_last_seen,status
124,best_house_26720423,Garage en Sant Cugat del Valles - Zona Coll Favà.,best_house,Sant Cugat del Vallès,45.0,4.500000e+01,2026-05-09T15:12:36+00:00,2026-05-09T15:12:36+00:00,0,0,active
120,tecnocasa_boxplaza-de-garaje-en-venta-585218,Box/plaza de garaje en venta,tecnocasa,Sant Cugat del Vallès,4500.0,4.500820e+05,2026-05-09T15:07:47+00:00,2026-05-09T15:08:17+00:00,0,0,active
11,qgat_homes_ref-1710,Garatge en venda i lloguer a Sant Cugat del Va...,qgat_homes,Sant Cugat del Vallès,8000.0,8.000000e+03,2026-05-09T11:22:01+00:00,2026-05-09T11:53:37+00:00,0,0,active
87,organ_ref-2008,PLAZA APARCAMIENTO VENTA,organ,Sant Cugat del Vallès,9000.0,9.000000e+03,2026-05-09T14:46:54+00:00,2026-05-09T14:47:21+00:00,0,0,active
85,organ_ref-1712,PARKING EN VENTA PLAZA LLUIS MILLET - CENTRO,organ,Sant Cugat del Vallès,10000.0,1.000000e+04,2026-05-09T14:46:54+00:00,2026-05-09T14:47:21+00:00,0,0,active
79,organ_ref-1011,GARAJE EN VENTA EN SANT FRANCESC-EL COLL - TOR...,organ,Sant Cugat del Vallès,10500.0,1.050000e+04,2026-05-09T14:46:54+00:00,2026-05-09T14:47:21+00:00,0,0,active
86,organ_ref-1853,PARKING EN VENTA AVENIDA LLUIS COMPANYS - CENTRO,organ,Sant Cugat del Vallès,10500.0,1.050000e+04,2026-05-09T14:46:54+00:00,2026-05-09T14:47:21+00:00,0,0,active
51,qgat_homes_ref-1519,Garatge en venda a Sant Cugat del Vallès,qgat_homes,Sant Cugat del Vallès,11000.0,1.100000e+04,2026-05-09T11:53:37+00:00,2026-05-09T11:53:37+00:00,0,0,active
80,organ_ref-1289,PARKING EN VENTA EN EL CENTRO - CENTRO,organ,Sant Cugat del Vallès,12000.0,1.200000e+04,2026-05-09T14:46:54+00:00,2026-05-09T14:47:21+00:00,0,0,active
78,fincas_moragas_0035-2,Garatge en venda a Volpelleres,fincas_moragas,Sant Cugat del Vallès,13000.0,3.513000e+303,2026-05-09T14:42:48+00:00,2026-05-09T14:43:38+00:00,0,0,active


## 7. Evolución de precio

Compara el primer precio visto con el precio actual. Útil para detectar bajadas y subidas.

In [17]:
if price_first_seen_col and price_first_seen_col in properties_df.columns:
    price_evolution_df = active_df.copy()
    price_evolution_df = price_evolution_df[price_evolution_df[price_first_seen_col].notna() & price_evolution_df['price'].notna()]
    price_evolution_df['price_delta'] = price_evolution_df['price'] - price_evolution_df[price_first_seen_col]
    price_evolution_df['price_delta_pct'] = (price_evolution_df['price_delta'] * 100.0 / price_evolution_df[price_first_seen_col]).round(2)
    display(price_evolution_df.sort_values(by='price_delta', key=lambda s: s.abs(), ascending=False).head(25)[[
        'property_id', 'title', 'source', 'city', 'price_first_seen', 'price', 'price_delta', 'price_delta_pct'
    ]])
else:
    print('No se encontró la columna price_first_seen en properties.')

,property_id,title,source,city,price_first_seen,price,price_delta,price_delta_pct
72,fincas_moragas_3994-2,Planta baixa en venda a Mira-sol,fincas_moragas,Sant Cugat del Vallès,inf,535000.0,-inf,NaN
75,fincas_moragas_6109-2,Local en venda a Volpelleres,fincas_moragas,Sant Cugat del Vallès,inf,2295000.0,-inf,NaN
74,fincas_moragas_3989-2,ESPLÈNDID ÀTIC A L'ARXIU,fincas_moragas,Sant Cugat del Vallès,inf,1195000.0,-inf,NaN
73,fincas_moragas_6111-2,Local en venda en Centre,fincas_moragas,Sant Cugat del Vallès,6.111882e+304,280000.0,-6.111882e+304,-100.00
78,fincas_moragas_0035-2,Garatge en venda a Volpelleres,fincas_moragas,Sant Cugat del Vallès,3.513000e+303,13000.0,-3.513000e+303,-100.00
77,fincas_moragas_4058-2,Terreny en venda en La Floresta,fincas_moragas,Sant Cugat del Vallès,4.058115e+302,2310000.0,-4.058115e+302,-100.00
76,fincas_moragas_0043-2,Garatge en venda en Volpelleres,fincas_moragas,Sant Cugat del Vallès,4.315217e+295,216900.0,-4.315217e+295,-100.00
111,tecnocasa_piso-en-venta-651578,Piso en venta,tecnocasa,Sant Cugat del Vallès,4.750005e+18,475000.0,-4.750005e+18,-100.00
118,tecnocasa_piso-en-venta-653116,Piso en venta,tecnocasa,Sant Cugat del Vallès,2.950003e+17,295000.0,-2.950003e+17,-100.00
116,tecnocasa_casa-en-venta-512221,Casa en venta,tecnocasa,Sant Cugat del Vallès,2.299001e+12,2299000.0,-2.298999e+12,-100.00


## 8. Vistas rápidas por tabla

Si quieres explorar una tabla concreta, descomenta una consulta o añade tus propios filtros aquí.

In [18]:
queries = {
    'active_properties': "SELECT property_id, source, title, price, price_first_seen, first_seen, last_seen, city, status FROM properties WHERE status = 'active' ORDER BY first_seen DESC",
    'oldest_active': "SELECT property_id, source, title, price, price_first_seen, first_seen, last_seen, city, status FROM properties WHERE status = 'active' ORDER BY first_seen ASC",
    'price_drops_or_changes': "SELECT property_id, source, title, price_first_seen, price, first_seen, last_seen FROM properties WHERE price_first_seen IS NOT NULL ORDER BY ABS(price - price_first_seen) DESC",
}

for name, sql in queries.items():
    print(f"\n### {name}")
    display(pd.read_sql_query(sql, conn).head(15))


### active_properties


,property_id,source,title,price,price_first_seen,first_seen,last_seen,city,status
0,aproperties_casa-de-422-m2-con-terraza-aparcam...,aproperties,Magnifica casa en venta en Eixample de Sant Cu...,1600000.0,1600000.0,2026-05-09T15:18:18+00:00,2026-05-09T15:18:18+00:00,Sant Cugat del Vallès,active
1,aproperties_piso-de-155-m2-con-terraza-aparcam...,aproperties,Piso en venta en Sant Cugat del Vallès,1040000.0,1040000.0,2026-05-09T15:18:18+00:00,2026-05-09T15:18:18+00:00,Sant Cugat del Vallès,active
2,aproperties_piso-de-155-m2-con-terraza-y-aparc...,aproperties,Piso en venta en Sant Cugat del Vallès,1090000.0,1090000.0,2026-05-09T15:18:18+00:00,2026-05-09T15:18:18+00:00,Sant Cugat del Vallès,active
3,aproperties_casa-de-242-m2-con-piscina-y-vista...,aproperties,Casa en venta en Valldoreix - Sant Cugat del V...,850000.0,850000.0,2026-05-09T15:18:18+00:00,2026-05-09T15:18:18+00:00,Sant Cugat del Vallès,active
4,aproperties_piso-de-128-m2-con-terraza-aparcam...,aproperties,Piso en venta en Sant Cugat,750000.0,750000.0,2026-05-09T15:18:18+00:00,2026-05-09T15:18:18+00:00,Sant Cugat del Vallès,active
5,aproperties_piso-de-213-m2-con-terraza-aparcam...,aproperties,Piso en venta en Can Trabal,1590000.0,1590000.0,2026-05-09T15:18:18+00:00,2026-05-09T15:18:18+00:00,Sant Cugat del Vallès,active
6,aproperties_casa-de-lujo-de-901-m2-con-terraza...,aproperties,Casa en venta en Masía Rosas,NaN,NaN,2026-05-09T15:18:18+00:00,2026-05-09T15:18:18+00:00,Sant Cugat del Vallès,active
7,aproperties_casa-de-500-m2-con-terraza-aparcam...,aproperties,Casa en venta en La Floresta,1400000.0,1400000.0,2026-05-09T15:18:18+00:00,2026-05-09T15:18:18+00:00,Sant Cugat del Vallès,active
8,aproperties_casa-de-412-m2-con-terraza-aparcam...,aproperties,Casa en venta en Sant Domenech,1800000.0,1800000.0,2026-05-09T15:18:18+00:00,2026-05-09T15:18:18+00:00,Sant Cugat del Vallès,active
9,aproperties_piso-de-132-m2-con-terraza-piscina...,aproperties,Piso en venta en Parc Central,780000.0,780000.0,2026-05-09T15:18:18+00:00,2026-05-09T15:18:18+00:00,Sant Cugat del Vallès,active



### oldest_active


,property_id,source,title,price,price_first_seen,first_seen,last_seen,city,status
0,amat_practica-i-accesible-planta-baixa-a-tocar...,amat,"Pràctica i accesible planta baixa, a tocar del...",310000.0,310000.0,2026-04-29T19:06:06+00:00,2026-05-09T14:21:20+00:00,Sant Cugat del Vallès,active
1,amat_fantastica-casa-de-disseny-a-mirasol,amat,Fantàstica casa de disseny a Mirasol,1225000.0,1225000.0,2026-04-29T19:06:06+00:00,2026-05-09T14:21:20+00:00,Sant Cugat del Vallès,active
2,amat_casa-a-sant-cugat-dos-habitatges-en-un,amat,Casa a Sant Cugat: Dos habitatges en un,845000.0,845000.0,2026-04-29T19:06:06+00:00,2026-04-29T19:08:50+00:00,Sant Cugat del Vallès,active
3,amat_placa-daparcament-en-venda-al-centre,amat,Plaça d'aparcament en venda al centre,20000.0,20000.0,2026-04-29T19:06:06+00:00,2026-05-09T14:21:20+00:00,Sant Cugat del Vallès,active
4,amat_magnifica-casa-al-golf-amb-grans-espais,amat,"Magnífica casa al Golf, amb grans espais",1575000.0,1575000.0,2026-04-29T19:06:06+00:00,2026-05-09T14:21:20+00:00,Sant Cugat del Vallès,active
5,qgat_homes_ref-1717,qgat_homes,Casa amb vistes a Mirasol,1150000.0,1150000.0,2026-05-09T11:22:01+00:00,2026-05-09T11:53:37+00:00,Sant Cugat del Vallès,active
6,qgat_homes_ref-1682,qgat_homes,Casa amb vistes a Mirasol,1150000.0,1150000.0,2026-05-09T11:22:01+00:00,2026-05-09T11:53:37+00:00,Sant Cugat del Vallès,active
7,qgat_homes_ref-1716,qgat_homes,Àtic en venda a Sant Cugat del Vallès - Can Mates,615000.0,615000.0,2026-05-09T11:22:01+00:00,2026-05-09T11:53:37+00:00,Sant Cugat del Vallès,active
8,qgat_homes_ref-1699,qgat_homes,Cases en venda a Sant Cugat del Vallès - Valld...,589000.0,589000.0,2026-05-09T11:22:01+00:00,2026-05-09T11:53:37+00:00,Sant Cugat del Vallès,active
9,qgat_homes_ref-1685,qgat_homes,Casa adossada a Sant Domenech,775000.0,775000.0,2026-05-09T11:22:01+00:00,2026-05-09T11:53:37+00:00,Sant Cugat del Vallès,active



### price_drops_or_changes


,property_id,source,title,price_first_seen,price,first_seen,last_seen
0,fincas_moragas_3994-2,fincas_moragas,Planta baixa en venda a Mira-sol,inf,535000.0,2026-05-09T14:42:48+00:00,2026-05-09T14:43:38+00:00
1,fincas_moragas_3989-2,fincas_moragas,ESPLÈNDID ÀTIC A L'ARXIU,inf,1195000.0,2026-05-09T14:42:48+00:00,2026-05-09T14:43:38+00:00
2,fincas_moragas_6109-2,fincas_moragas,Local en venda a Volpelleres,inf,2295000.0,2026-05-09T14:42:48+00:00,2026-05-09T14:43:38+00:00
3,fincas_moragas_6111-2,fincas_moragas,Local en venda en Centre,6.111882e+304,280000.0,2026-05-09T14:42:48+00:00,2026-05-09T14:43:38+00:00
4,fincas_moragas_0035-2,fincas_moragas,Garatge en venda a Volpelleres,3.513000e+303,13000.0,2026-05-09T14:42:48+00:00,2026-05-09T14:43:38+00:00
5,fincas_moragas_4058-2,fincas_moragas,Terreny en venda en La Floresta,4.058115e+302,2310000.0,2026-05-09T14:42:48+00:00,2026-05-09T14:43:38+00:00
6,fincas_moragas_0043-2,fincas_moragas,Garatge en venda en Volpelleres,4.315217e+295,216900.0,2026-05-09T14:42:48+00:00,2026-05-09T14:43:38+00:00
7,tecnocasa_piso-en-venta-651578,tecnocasa,Piso en venta,4.750005e+18,475000.0,2026-05-09T15:07:47+00:00,2026-05-09T15:08:17+00:00
8,tecnocasa_piso-en-venta-653116,tecnocasa,Piso en venta,2.950003e+17,295000.0,2026-05-09T15:07:47+00:00,2026-05-09T15:08:17+00:00
9,tecnocasa_casa-en-venta-512221,tecnocasa,Casa en venta,2.299001e+12,2299000.0,2026-05-09T15:07:47+00:00,2026-05-09T15:08:17+00:00
